# Premise + Imported LCI (Tutorial)

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

## Purpose

This notebook demonstrates a realistic TRAILS workflow:

1. Load a premise datapackage.
2. Inspect the available scenario years and annual interpolation.
3. Import an Excel foreground inventory.
4. Select the foreground activity by metadata instead of by a hard-coded index.
5. Inspect temporal exchange settings.
6. Run temporal LCA, static LCA, and compare their scores directly.
7. Plot temporal scores and connect the result to the FaIR climate emulator.

## Prerequisites

- A premise datapackage generated in Notebook 2.1, or another compatible
  TRAILS datapackage.
- The example foreground inventory `lci-pass_cars.xlsx`.

## Theory (short)

TRAILS (Temporal Routing And Aggregation of Impacts across Life-cycle Systems)
makes time explicit in LCA. Exchanges can carry temporal distributions that
spread burdens across future or past years. Temporal LCA therefore gives a time
series of impacts, while static LCA gives a single-year benchmark for a chosen
assessment year.


In [ ]:
import logging
from pathlib import Path

import pandas as pd
from datapackage import Package

from trails import (
    Trails,
    plot_temporal_scores,
    plot_rf,
    plot_temp,
    search_activity,
)
from trails.logging import configure_trails_logging


## 1. Load the premise data package

Point `premise_path` to the zip file exported in Notebook 2.1. The example
inventory file is resolved relative to either the current directory or the
repository `examples/` folder.


In [ ]:
configure_trails_logging(level=logging.INFO)

premise_path = Path("/path/to/your/premise-datapackage.zip")
inventory_path = Path("lci-pass_cars.xlsx")
if not inventory_path.exists():
    inventory_path = Path("examples") / "lci-pass_cars.xlsx"

if not premise_path.exists():
    raise FileNotFoundError(
        "Set `premise_path` to the datapackage zip created in Notebook 2.1."
    )
if not inventory_path.exists():
    raise FileNotFoundError("Could not find `lci-pass_cars.xlsx`.")

methods = [
    "IPCC 2021 - climate change: total (excl. biogenic CO2) - global warming potential (GWP100)",
    "IPCC 2021 (incl. biogenic CO2) - climate change: total (incl. biogenic CO2) - global warming potential (GWP100)",
]

package = Package(str(premise_path))
trails = Trails(
    package=package,
    interpolate_annual=True,
    methods=methods,
    ei_version="3.12",
)


## 2. Inspect scenario years and annual interpolation

`template_labels` are the original scenario years exported by premise.
`scenario_labels` become annual when `interpolate_annual=True`.

This is the practical link back to Notebook 2.1: exporting five anchor years is
fine for a tutorial, but the annual years seen here come from TRAILS
interpolation after loading.


In [ ]:
print("Template scenario years:", trails.template_labels)
print("Interpolated year range:", trails.min_year, "to", trails.max_year)
print("Number of years after interpolation:", len(trails.scenario_labels))
print("A shape:", trails.A.shape)
print("B shape:", trails.B.shape)


## 3. Choose LCIA methods

The order of `methods` matters because the static score is returned in the same
order.


In [ ]:
# These methods are configured on the Trails instance.
methods = trails.methods
methods


## 4. Import the Excel foreground inventory

This merges the Excel inventory into the loaded datapackage. Year-specific
columns in the spreadsheet are written into the corresponding matrix years and
TRAILS interpolates between them when needed.


In [ ]:
trails.import_excel_inventory(str(inventory_path))


## 5. Select the foreground activity by metadata

Do not hard-code an activity index. The integer `idx` can change across
machines or datapackages even when the activity metadata stays the same.

The cell below searches for the diesel passenger car activity, converts the
result table to pandas, and extracts the matching `index` explicitly.


In [ ]:
ref_year = 2050
search_name = "transport, passenger, car,"
full_name = "transport, passenger, car, diesel"
preferred_location = None  # Set this if you need to narrow multiple exact matches.

matches = search_activity(trails, search_name)
matches_df = pd.DataFrame(matches.rows, columns=matches.field_names)

candidate_matches = matches_df[
    matches_df["name"].str.contains("diesel", case=False, na=False)
].copy()
candidate_matches

exact_matches = matches_df[matches_df["name"] == full_name].copy()
if preferred_location is not None:
    exact_matches = exact_matches[exact_matches["location"] == preferred_location]

if len(exact_matches) != 1:
    raise ValueError(
        "Expected exactly one matching activity. Refine `full_name` and/or `preferred_location`."
    )

idx = int(exact_matches["index"].item())
print("Selected activity index:", idx)
exact_matches


## 6. Inspect exchanges and temporal distribution settings

`trails.print_exchange_table(...)` prints the exchanges for the selected
activity and also shows the temporal distribution fields used by TRAILS.


In [ ]:
trails.print_exchange_table(year=ref_year, act_idx=idx)


The temporal distribution codes in the exchange table mean:

- `1`: discrete (all mass at `temporal_loc`)
- `2`: lognormal
- `3`: normal
- `4`: uniform
- `5`: triangular
- `6`: discrete empirical (explicit pulses from `temporal_offsets` and `temporal_weights`)

Codes `2` to `6` are the ones you will typically want to explain when reading a
foreground exchange table, because they indicate how an exchange is spread over
time instead of occurring in a single pulse.


In [ ]:
temporal_distribution_reference = pd.DataFrame(
    [
        {"code": 1, "distribution": "discrete", "meaning": "all mass at temporal_loc"},
        {"code": 2, "distribution": "lognormal", "meaning": "right-skewed spread over time"},
        {"code": 3, "distribution": "normal", "meaning": "symmetric spread around temporal_loc"},
        {"code": 4, "distribution": "uniform", "meaning": "equal weight between temporal_min and temporal_max"},
        {"code": 5, "distribution": "triangular", "meaning": "bounded spread with peak at temporal_loc"},
        {"code": 6, "distribution": "discrete empirical", "meaning": "explicit pulses from offsets and weights"},
    ]
)
temporal_distribution_reference


## 7. Temporal routing and graph visualization

Routing shows how demand propagates through time before the full LCA solve.


In [ ]:
from trails.plotting import plot_temporal_graph

trails.temporal_routing(
    start_year=ref_year,
    start_act_idx=idx,
    amount=1.0,
    # adaptive routing is the default: max_depth=None, relative cutoff=1e-4
    show_progress=True,
    attribute_to_roots=True,
)

plot_temporal_graph(
    trails,
    filename="trails_graph.html",
    notebook=False,
)


## 8. Temporal LCA (full solve)

This computes time-resolved scores and stores the inventory for downstream
analysis.

The temporal LCA result is available on `trails.scores`. Because that object can
include several dimensions (method, activity, year, and sometimes root
activity), the next comparison step will collapse every non-method dimension to
recover one total score per LCIA method.


In [ ]:
trails.lca(
    show_progress=True,
    compute_score=True,
    store_inventory=True,
)


## 9. Static LCA (single-year benchmark)

Here, **static** means that the system is solved as a single-year LCA for the
chosen assessment year `ref_year`. It does **not** mean "present-day only".

A static LCA can still be prospective or retrospective depending on the year and
scenario datapackage you choose. In this notebook, a static calculation at 2050
is still a prospective result, but it is evaluated as one single-year system
instead of spreading exchanges over time.


In [ ]:
trails.static_lca(
    year=ref_year,
    act_idx=idx,
)

static_score = trails.static_score
static_score


## 10. Compare temporal and static scores directly

This cell makes the temporal LCA total explicit, so you can compare it with the
static benchmark before looking at plots.


In [ ]:
temporal_score = trails.scores
for dim in [dim for dim in temporal_score.dims if dim != "method"]:
    temporal_score = temporal_score.sum(dim=dim)

temporal_score_values = temporal_score.to_numpy().astype(float).ravel()
temporal_score_series = pd.Series(
    temporal_score_values,
    index=pd.Index(methods, name="method"),
    name="temporal_lca_score",
)

static_score_values = static_score if isinstance(static_score, list) else [static_score]
static_score_series = pd.Series(
    [float(value) for value in static_score_values],
    index=pd.Index(methods, name="method"),
    name="static_lca_score",
)

score_comparison = pd.concat([temporal_score_series, static_score_series], axis=1)
score_comparison["difference"] = (
    score_comparison["temporal_lca_score"] - score_comparison["static_lca_score"]
)
score_comparison


## 11. Plot temporal scores

The red dotted line overlays the static score on the temporal plot.


In [ ]:
figs = plot_temporal_scores(
    trails=trails,
    stacked=False,
    legend_top_n=7,
    show_flow_contributions=False,
    title="",
    method_label="kg CO2-eq",
    cumulative=False,
    width=550,
    height=450,
    year_tick=5,
    year_range=(2000, 2100),
    reference_year=ref_year,
    show_cumulative_axis=True,
    static_score=static_score,
    static_score_dash="dot",
    static_score_color="red",
)


In [ ]:
figs[0]


In [ ]:
figs[1]


## 12. Climate emulator (FaIR)

Because `store_inventory=True` was used in `trails.lca(...)`, the characterized
inventory is available for the downstream radiative forcing and temperature
steps.


In [ ]:
from trails.fair_rf import run_fair_delta_rf

run_fair_delta_rf(trails)


### Plot radiative forcing and temperature


In [ ]:
fig = plot_rf(
    trails=trails,
    reference_year=ref_year,
    by="flow",
    year_range=(2000, 2150),
    year_tick=10,
)
fig


In [ ]:
plot_temp(
    trails,
    by="root activity",
    title="Temperature change by root activity",
    method_label="degC",
    year_range=(2000, 2150),
)
